# Import

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import classification_report
import lightgbm as lgb
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# Add src/ to path (once, so imports work)
sys.path.append(str(Path().resolve().parent / "src"))
# 
# Enable autoreload for Jupyter notebooks
%load_ext autoreload
%autoreload 2

from paths import DATA_DATASETS
from helper_functions import get_master_dataframe
import feature_construction as fc

Failed to read module file 'C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\User\Documents\ETH-repositories\Marketing-Analytics\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "c:\Users\User\Documents\ETH-repositories\Marketing-Analytics\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1398, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib._b

In [2]:
# Get master dataframe - constructed of transactions, outfits, and outfit_clusters
master = get_master_dataframe()


test_customers = pd.read_csv(DATA_DATASETS / "test_customers.csv", sep=";")
baseline_pred = pd.read_csv(DATA_DATASETS / "pred.csv")

Master shape: (60897, 11)
   customer.id                                outfit.id rentalPeriod.start  \
0         3448  outfit.5c081909537b42239e465d2d615c705f         2023-03-26   
1         2924  outfit.c34969dd8b334064aa90bfb60c8ec308         2023-03-27   

  rentalPeriod.end  cluster         cluster_name  pricePerWeek  pricePerMonth  \
0       2023-04-25      6.0  Sweaters & Knitwear         750.0         1500.0   
1       2023-04-26      4.0   Trousers & Jackets         990.0         1980.0   

   retailPrice  duration_days  revenue  
0       2500.0             30   1500.0  
1       3900.0             30   1980.0  


In [3]:
# Fold structure
# TIMEFRAME_START  = pd.Timestamp("2017-04-01")
TRAIN_CUTOFF     = pd.Timestamp("2021-09-06")  # features end here for training
LABEL_END        = pd.Timestamp("2022-09-06")  # labels end here for training
FINAL_CUTOFF     = pd.Timestamp("2023-09-06")  # features end here for submission

X_train, X_test, y_train, y_test, id_train, id_test = fc.generate_train_test_splits(df=master, train_cutoff=TRAIN_CUTOFF, label_end=LABEL_END, final_cutoff=FINAL_CUTOFF)

In [4]:
# Feature Selection: Sequential Forward Selection (SFS) with LightGBM for the Baseline Model
sfs_selected_features = ['recency', 'monetary', 'avg_revenue', 'weighted_rev', 'monetary_x_frequency', 'tenure_days', 'revenue_per_day', 'rentals_per_day', 'n_summer', 'active_months', 'revenue_trend', 'pct_weekly_rentals', 'cluster_affinity_0', 'cluster_affinity_1', 'cluster_affinity_2', 'cluster_affinity_7', 'max_retail_price', 'avg_price_per_week']

X_train_sfs = X_train[sfs_selected_features]
X_test_sfs = X_test[sfs_selected_features]

# 2-stage model
# Feature Selection: Sequential Forward Selection (SFS) with LightGBM for the Churn Model
sfs_selected_features_2s_clf = [
    'recency',
    'monetary_x_frequency',
    'pct_weekly_rentals',
    'cluster_affinity_6'
]

X_train_sfs_clf = X_train[sfs_selected_features_2s_clf]
X_test_sfs_clf = X_test[sfs_selected_features_2s_clf]

# Feature Selection: Sequential Forward Selection (SFS) with LightGBM for the Regression Model
# sfs_selected_features_2s_reg = ['recency', 'monetary', 'tenure_days', 'avg_days_between_rentals', 'revenue_trend']
sfs_selected_features_2s_reg = [
    'recency', 
    'monetary', 
    'avg_revenue', 
    'weighted_rev', 
    'monetary_x_frequency', 
    'tenure_days', 
    'revenue_per_day', 
    'rentals_per_day', 
    'revenue_trend', 
    'max_retail_price', 
    'avg_price_per_week'
]

X_train_sfs_reg = X_train[sfs_selected_features_2s_reg]
X_test_sfs_reg = X_test[sfs_selected_features_2s_reg]

In [5]:
# Fit and evaluate
lgbm_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1,

    objective="tweedie",
    tweedie_variance_power=1.3
)

# Train fold
lgbm_model.fit(X_train_sfs, y_train)

train_predictions = np.clip(np.array(lgbm_model.predict(X_train_sfs)), 0, None)
train_mae = mean_absolute_error(y_train, train_predictions)
print(f"MAE on training fold: {train_mae:.2f} NOK")


# Test fold
test_predictions = np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)

test_mae = mean_absolute_error(y_test, test_predictions)
print(f"MAE on test fold: {test_mae:.2f} NOK")

MAE on training fold: 370.89 NOK
MAE on test fold: 2568.38 NOK


## Two-stage Model

In [6]:
# Stage 1: Who is going to be active?
y_train_churn = (y_train > 0).astype(int).values.ravel()
y_test_churn  = (y_test  > 0).astype(int).values.ravel()

n_inactive = (y_train_churn == 0).sum()
n_active   = (y_train_churn == 1).sum()
print(f"Training: {n_active} aktiv, {n_inactive} inaktiv ({n_inactive/n_active:.1f}:1)")

churn_model = lgb.LGBMClassifier(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    # scale_pos_weight  = n_inactive / n_active,
    random_state      = 42,
    verbose           = -1
)
churn_model.fit(X_train_sfs_clf, y_train_churn)

churn_prob_test  = churn_model.predict_proba(X_test_sfs_clf)[:, 1]
churn_pred_test  = churn_model.predict(X_test_sfs_clf)

print("Churn Classifier — Test Fold:")
print(classification_report(y_test_churn, churn_pred_test, target_names=["Inactive", "Active"]))



# Stage 2: How much revenue do active customers generate?
active_mask = (y_train > 0).values.ravel()

print(f"\nRevenue Model trained on {active_mask.sum()} active customers.")

# Log-transform the target for better modeling (optional, but often helps with skewed revenue data)
# y_train_log = np.log1p(y_train.values.ravel()[active_mask])

revenue_model = lgb.LGBMRegressor(
    n_estimators      = 600,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 0,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1,

    objective="tweedie",
    tweedie_variance_power=1.8
)

# revenue_model.fit(X_train[active_mask], y_train_log)
revenue_model.fit(X_train_sfs_reg[active_mask], y_train.values.ravel()[active_mask])
# revenue_model.fit(X_train_sfs_reg[active_mask], y_train_log)

# Predict and transform back from log scale
pred_log = np.array(revenue_model.predict(X_test_sfs_reg))
revenue_if_active = np.clip(pred_log, 0, None)
#revenue_if_active = np.clip(np.expm1(pred_log), 0, None)

# Hard: entweder 0 oder predicted revenue
final_pred_hard = np.where(churn_pred_test == 1, revenue_if_active, 0)

# Soft: P(aktiv) × expected revenue — oft besser für MAE
final_pred_soft = churn_prob_test * revenue_if_active

mae_hard      = mean_absolute_error(y_test, final_pred_hard)
mae_soft      = mean_absolute_error(y_test, final_pred_soft)
mae_baseline  = mean_absolute_error(y_test, np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None))

print(f"\nResults on Test Fold:")
print(f"Single-Stage LightGBM: {mae_baseline:.2f} NOK")
print(f"Two-Stage Hard: {mae_hard:.2f} NOK")
print(f"Two-Stage Soft: {mae_soft:.2f} NOK")

Training: 439 aktiv, 5504 inaktiv (12.5:1)
Churn Classifier — Test Fold:
              precision    recall  f1-score   support

    Inactive       0.98      0.97      0.98      6193
      Active       0.74      0.79      0.76       616

    accuracy                           0.96      6809
   macro avg       0.86      0.88      0.87      6809
weighted avg       0.96      0.96      0.96      6809


Revenue Model trained on 439 active customers.

Results on Test Fold:
Single-Stage LightGBM: 2568.38 NOK
Two-Stage Hard: 2589.89 NOK
Two-Stage Soft: 2748.89 NOK


In [7]:
y_test_flat = y_test.values.ravel()

# Diagnostics
results = pd.DataFrame({
    "actual":         y_test_flat,
    "pred_single":    np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None),
    "pred_hard":      final_pred_hard,
    "pred_soft":      final_pred_soft,
    "churn_prob":     churn_prob_test,
    "error_single":   np.abs(y_test_flat - np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)),
    "error_hard":     np.abs(y_test_flat - final_pred_hard),
    "error_soft":     np.abs(y_test_flat - final_pred_soft),
})

results["bucket"] = pd.cut(results["actual"],
    bins=[-0.01, 0.01, 1000, 5000, 15000, 30000, 45000, 60000, 999999],
    labels=["Churned (0)", "1–1k NOK", "1k–5k NOK", "5k-15k NOK", "15k-30k NOK", "30k-45k NOK", "45k-60k NOK", "60k+ NOK"])

print("\nMAE for each customer segment:")
print(results.groupby("bucket", observed=True).agg(
    n           = ("actual",      "count"),
    mae_single  = ("error_single","mean"),
    mae_hard    = ("error_hard",  "mean"),
    mae_soft    = ("error_soft",  "mean"),
    avg_actual  = ("actual",      "mean"),
).round(2).to_string())

# Threshold Optimisation
# Find optimal threshold for churn classifier to minimize MAE
print("\nThreshold Optimization:")
thresholds = np.arange(0.5, 1, 0.01)
threshold_results = []

for t in thresholds:
    pred = np.where(churn_prob_test >= t, revenue_if_active, 0)
    mae  = mean_absolute_error(y_test, pred)
    threshold_results.append({"threshold": t, "mae": mae})

thresh_df = pd.DataFrame(threshold_results)
best_thresh = thresh_df.loc[thresh_df["mae"].idxmin(), "threshold"]
best_mae    = thresh_df["mae"].min()

print(thresh_df.to_string(index=False))
print(f"\nBest Threshold: {best_thresh:.2f} → MAE: {best_mae:.2f} NOK")


MAE for each customer segment:
                n  mae_single  mae_hard  mae_soft  avg_actual
bucket                                                       
Churned (0)  6193      417.16    477.10    654.02        0.00
1k–5k NOK      84    10077.82  12681.18  11942.08     2951.35
5k-15k NOK    122    12291.70  13509.21  12368.40     9855.13
15k-30k NOK   103    15795.19  16866.84  16599.51    21818.84
30k-45k NOK   102    22738.50  17695.78  18370.83    38050.78
45k-60k NOK    96    31191.69  26544.15  27187.90    51650.07
60k+ NOK      109    51539.78  53908.40  54689.81    96566.65

Threshold Optimization:
 threshold         mae
      0.50 2589.885821
      0.51 2570.223626
      0.52 2565.645237
      0.53 2566.541834
      0.54 2547.332703
      0.55 2545.426448
      0.56 2543.935137
      0.57 2549.925595
      0.58 2544.588152
      0.59 2544.009317
      0.60 2537.489708
      0.61 2531.000429
      0.62 2527.925338
      0.63 2525.169959
      0.64 2528.404159
      0.65 2529.7

In [10]:
# --- FINAL EXPORT: CHURN PREDICTION ---

# 1. Generate features for the final prediction window (up to Sept 6, 2023)
# We shift the timeframes forward by one period to get the current customer state
_, X_submission_raw, _, _, _, id_submission = fc.generate_train_test_splits(
    df=master, 
    train_cutoff=LABEL_END,                     
    label_end=FINAL_CUTOFF,                     
    final_cutoff=pd.Timestamp("2024-09-06")     
)

# 2. Select the specific features used by the trained churn model
X_submission_clf = X_submission_raw[sfs_selected_features_2s_clf]

# 3. Predict active status (Model outputs 1 = Active, 0 = Churned)
pred_active = churn_model.predict(X_submission_clf)

# 4. Invert the labels to match assignment requirements (1 = Churned, 0 = Active)
pred_churn = 1 - pred_active

# 5. Create a mapping dataframe for safe merging
predictions_df = pd.DataFrame({
    'join_id': id_submission.values.ravel() if hasattr(id_submission, 'values') else id_submission,
    'predicted_churn': pred_churn
})

# 6. Load the original test_customers template
test_customers_export = pd.read_csv(DATA_DATASETS / "test_customers.csv", sep=";")

# 7. Merge predictions reliably
id_col = 'customer_id' if 'customer_id' in test_customers_export.columns else 'customer.id'

test_customers_export = test_customers_export.drop(columns=['churn_23_24'], errors='ignore')
test_customers_export = test_customers_export.merge(
    predictions_df, left_on=id_col, right_on='join_id', how='left'
)

# 8. Clean up and populate the target column
test_customers_export['churn_23_24'] = test_customers_export['predicted_churn']
test_customers_export = test_customers_export.drop(columns=['predicted_churn', 'join_id'], errors='ignore')

# 9. Save the final file
test_customers_export.to_csv(DATA_DATASETS / "test_customers_FINAL.csv", sep=";", index=False)

print("Success! Churn predictions generated, inverted (1=Churned, 0=Active), and saved to 'test_customers_FINAL.csv'")

Success! Churn predictions generated, inverted (1=Churned, 0=Active), and saved to 'test_customers_FINAL.csv'
